# CANDIDATE FEATURE IDENTIFICATION
The purpose of this notebook is to aggregate in one location the potential features to include in the training set. 
Inputs: External Dataset Parquet Files. 
Desired Outputs: 
- Of redundant variables, select the most reliable and informative one. 
- Understand how to calculate any computed features.

Notes
- Use a separate .py module under src/external_dataset_name/ingest.py to pull in the data with API or other method and write to data/interim/external_dataset_name.parquet. 
- This notebook should read in just a small sample of each dataset for feature exploration. Reading in the full dataset may crash the Jupyter kernel. 

For each feature, attempt to identify:
- time of observation
- unit of observation / what it actually is
- geographic scope (i.e. is it at the census block group, tract, or zip code level?)
- missingness
- what preprocessing is required (e.g. geocoding to census block group, converting a raw count to density)

In [6]:
from pathlib import Path

import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq

In [2]:
pd.set_option('display.max_columns', None)

## Lottery and Sales Data (for reference when merging)

In [3]:
lottery_locations_file_path = Path(".").resolve().parent / "data/final/location_data.parquet"

#locations_pq = pq.ParquetFile(lottery_locations_file_path)
#locations_first_100_rows = next(locations_pq.iter_batches(batch_size=100))
#locations_df = pa.Table.from_batches([locations_first_100_rows]).to_pandas()

locations_df = pd.read_parquet(str(lottery_locations_file_path))

In [5]:
locations_df.sample(5)

,agtno,busnm,busaddr,buscity,buszip,chain,bustype,longitude,latitude,geometry,bg_geoid,tract_geoid
2290,111049,M & L FOOD CENTER CORP,131 TERRACE VIEW AVE STORE 1,BRONX,10463,0,SUPERMARKETS,-73.91289,40.87597,b'\x01\x01\x00\x00\x00\x1c\xb6-\xcalzR\xc0y#\x...,360610309005,36061030900
10046,102330,SHREE GANESH CARDS INC,4258 HICKSVILLE RD,BETHPAGE,11714,0,STATIONERY-CARD,-73.48393,40.71709,b'\x01\x01\x00\x00\x00i\xe3\x88\xb5\xf8^R\xc0\...,360595203004,36059520300
10290,103360,PAPASITO FOOD MARKET,3605 BROADWAY,NEW YORK,10031,0,DELI-GROCERY,-73.94893,40.82866,b'\x01\x01\x00\x00\x00_\x0c\xe5D\xbb|R\xc0n\xc...,360610233001,36061023300
9777,101271,RT 24 DELI,1820 HEMPSTEAD TPKE,EAST MEADOW,11554,0,CONV. STORE,-73.56913,40.72369,b'\x01\x01\x00\x00\x00\nK<\xa0ldR\xc0\x9b8\xb9...,360594080003,36059408000
2115,102653,ELITE WINE & SPIRIT LLC,231A E 149TH ST,BRONX,10451,0,LIQUOR,-73.92437,40.81782,b'\x01\x01\x00\x00\x00\xd3\xd9\xc9\xe0({R\xc0\...,360050065002,36005006500


In [12]:
lottery_sales_file_path = Path(".").resolve().parent / "data/final/sales_data.parquet"

#sales_pq = pq.ParquetFile(lottery_sales_file_path)
#sales_first_100_rows = next(sales_pq.iter_batches(batch_size=1000))
#sales_df = pa.Table.from_batches([sales_first_100_rows]).to_pandas()

sales_df = pd.read_parquet(str(lottery_sales_file_path))

In [13]:
sales_df.sample(5)

,agtno,total_sales,first_date,last_date,first_pos_date,last_pos_date,pos_sales_days,num_cal_days,num_active_days,days_before_first_sale,days_after_last_sale,avg_cal_daily_sales,avg_active_daily_sales,use_active_period,avg_daily_sales
6036,114629,16785.0,2025-01-01,2025-12-31,2025-01-01,2025-12-31,235.0,365,365.0,0.0,0.0,45.986301,45.986301,False,45.986301
6634,115334,1320195.5,2025-01-01,2025-12-31,2025-01-01,2025-12-31,365.0,365,365.0,0.0,0.0,3616.973973,3616.973973,False,3616.973973
4157,111724,244757.5,2025-01-01,2025-12-31,2025-01-01,2025-12-31,364.0,365,365.0,0.0,0.0,670.568493,670.568493,False,670.568493
7096,115828,236797.0,2025-05-19,2025-12-31,2025-05-19,2025-12-31,227.0,227,227.0,0.0,0.0,1043.158590,1043.158590,False,1043.158590
284,101214,1441717.0,2025-01-01,2025-12-31,2025-01-01,2025-12-31,365.0,365,365.0,0.0,0.0,3949.909589,3949.909589,False,3949.909589


## Overture Maps Data

In [ ]:
overture_file_path = Path(".").resolve().parent / "data/interim/external/merged_with_loc/location_data_overture.parquet"

overture_pq = pq.ParquetFile(overture_file_path)
overture_first_100_rows = next(overture_pq.iter_batches(batch_size=100))
overture_df = pa.Table.from_batches([overture_first_100_rows]).to_pandas()

In [15]:
overture_df.sample(5)

,agtno,busnm,busaddr,buscity,buszip,longitude,latitude,geometry,bg_geoid,tract_geoid,gas_station_within_0.5_mi,gas_station_within_1_mi,arts_and_entertainment_within_0.5_mi,arts_and_entertainment_within_1_mi,shopping_within_0.5_mi,shopping_within_1_mi,social_or_community_service_within_0.5_mi,social_or_community_service_within_1_mi,restaurant_within_0.5_mi,restaurant_within_1_mi,bar_within_0.5_mi,bar_within_1_mi
92,76475,QUICKLEES #12,349 W MORRIS ST,BATH,14810,-77.33299,42.33839,b'\x01\x01\x00\x00\x00H\xf9I\xb5OUS\xc0\x8f\xa...,361019611001,36101961100,2,2,0,0,0,0,1,9,0,1,0,2
88,102050,FELIO FOOD CORP,1161 SUTTER AVE,BROOKLYN,11208,-73.87513,40.67232,b'\x01\x01\x00\x00\x00\xe7oB!\x02xR\xc0.9\xee\...,360471194003,36047119400,3,10,1,5,4,8,4,14,2,10,0,3
74,111311,ADK FOOD & FUEL,1 COUNTRY MILE,FRANKFORT,13340,-75.08829,43.03850,b'\x01\x01\x00\x00\x00\x16\xa4\x19\x8b\xa6\xc5...,360430102022,36043010202,0,2,1,1,0,0,0,0,0,0,0,1
50,88721,A TO Z GROCERY & DELI INC,1450 CROSBY AVE,BRONX,10461,-73.83000,40.84207,b'\x01\x01\x00\x00\x00\x85\xebQ\xb8\x1euR\xc0\...,360050264001,36005026400,4,10,0,1,1,4,7,18,1,6,3,6
96,95155,ARIAS LIQUOR INC,1648 BRUCKNER BLVD,BRONX,10473,-73.87382,40.82335,b'\x01\x01\x00\x00\x00\xce\xa5\xb8\xaa\xecwR\x...,360050028001,36005002800,4,16,1,4,2,8,3,17,5,13,1,1


## LEHD Data

In [ ]:
rac_file_path = Path(".").resolve().parent / "data/interim/external/merged_with_loc/location_data_rac.parquet"

rac_pq = pq.ParquetFile(rac_file_path)
rac_first_100_rows = next(rac_pq.iter_batches(batch_size=100))
rac_df = pa.Table.from_batches([rac_first_100_rows]).to_pandas()

In [17]:
rac_df.sample(2)

,agtno,busnm,busaddr,buscity,buszip,longitude,latitude,geometry,bg_geoid,tract_geoid,rac_c000,rac_ce01,rac_ce02,rac_ce03
40,68056,7-ELEVEN #16361E,33 MIDDLE COUNTRY RD RTE 25,CORAM,11727,-72.97863,40.87459,b'\x01\x01\x00\x00\x00\x9b8\xb9\xdf\xa1>R\xc0N...,361031583091,36103158309,917.0,119.0,247.0,551.0
88,48170,DELHI DINER LLC,95 MAIN ST,DELHI,13753,-74.91717,42.27689,b'\x01\x01\x00\x00\x00\xd4\xb7\xcc\xe9\xb2\xba...,360259708004,36025970800,341.0,66.0,80.0,195.0


In [ ]:
wac_file_path = Path(".").resolve().parent / "data/interim/external/merged_with_loc/location_data_wac.parquet"

wac_pq = pq.ParquetFile(wac_file_path)
wac_first_100_rows = next(wac_pq.iter_batches(batch_size=100))
wac_df = pa.Table.from_batches([wac_first_100_rows]).to_pandas()

In [19]:
wac_df.sample(2)

,agtno,busnm,busaddr,buscity,buszip,longitude,latitude,geometry,bg_geoid,tract_geoid,wac_c000,wac_cns07,wac_cns18,wac_ce01,wac_ce02,wac_ce03
64,81101,SOFT TOUCH NEWSTAND,1 METROTECH CENTER,BROOKLYN,11201,-73.98723,40.69376,b'\x01\x01\x00\x00\x00S\xe8\xbc\xc6.\x7fR\xc0\...,360470011001,36047001100,21611.0,171.0,872.0,2036.0,2591.0,16984.0
94,76475,QUICKLEES #12,349 W MORRIS ST,BATH,14810,-77.33299,42.33839,b'\x01\x01\x00\x00\x00H\xf9I\xb5OUS\xc0\x8f\xa...,361019611001,36101961100,1017.0,265.0,169.0,202.0,322.0,493.0


## AADT Data

In [ ]:
aadt_file_path = Path(".").resolve().parent / "data/interim/external/merged_with_loc/data_aadt_2025.parquet"

aadt_pq = pq.ParquetFile(aadt_file_path)
aadt_first_100_rows = next(aadt_pq.iter_batches(batch_size=100))
aadt_df = pa.Table.from_batches([aadt_first_100_rows]).to_pandas()

In [7]:
aadt_df.sample(5)

NameError: name 'aadt_df' is not defined

## EPA Data

In [8]:
epa_file_path = Path(".").resolve().parent / "data/interim/external/merged_with_loc/location_data_epa.parquet"

epa_pq = pq.ParquetFile(epa_file_path)
epa_first_100_rows = next(epa_pq.iter_batches(batch_size=100))
epa_df = pa.Table.from_batches([epa_first_100_rows]).to_pandas()

In [9]:
epa_df.sample(5)

,agtno,busnm,busaddr,buscity,buszip,chain,bustype,longitude,latitude,geometry,bg_geoid,tract_geoid,OBJECTID,STATEFP,COUNTYFP,TRACTCE,BLKGRPCE,TotPop,HH,R_LowWageWk,R_MedWageWk,TotEmp,E_LowWageWk,E_MedWageWk,E_HiWageWk,D1A,D1B,D1C,D1C5_RET,D1C5_OFF,D1C5_IND,D1C5_SVC,D1C5_ENT,D1C8_RET,D1C8_OFF,D1C8_IND,D1C8_SVC,D1C8_ENT,D1C8_ED,D1C8_HLTH,D1C8_PUB,D3A,D3B,D3BAO,D3BMM3,D3BMM4,D3BPO3,D3BPO4,D4A,NatWalkInd,STATEFP2,COUNTYFP2,TRACTCE2,BLKGRPCE2
56,110788,SUNRISE 99 CENT STORE INC,4008 QUEENS BLVD,SUNNYSIDE,11104,0,VARIETY,-73.92392,40.74361,b'\x01\x01\x00\x00\x00\xb3\x07Z\x81!{R\xc0A}\x...,360810181013,36081018101,162297,36,81,18101,3,1302,632.0,118,237,316,58,83,175,73.247865,137.816070,33.448447,2.857937,2.857937,24.345389,2.116990,1.270194,2.857937,2.857937,24.345389,1.376044,1.270194,0.529248,0.211699,0.0,39.499071,67.743690,67.74369,0.000000,67.743690,0.0,0.000000,320.53,12.500000,36,081,018101,3
85,113520,LILI'S DELI GROCERY CORP,995 KELLY ST STORE 16,BRONX,10459,0,SUPERMARKETS,-73.89586,40.82283,b'\x01\x01\x00\x00\x00\xdar.\xc5UyR\xc0w\x15R~...,360050129011,36005012901,153507,36,5,12901,1,1279,372.0,146,285,574,70,130,374,21.865481,65.037093,29.187874,1.525499,0.355950,0.559350,24.611378,2.135698,1.525499,0.355950,0.559350,1.576349,2.135698,21.001032,2.033998,0.0,30.121995,108.501601,0.00000,65.087943,32.543972,0.0,32.543972,238.72,14.833333,36,005,012901,1
46,91274,KINGS NAILS EIGHT LLC,3622 WHITE PLAINS RD,BRONX,10467,0,SPECIALTY(NON FOOD),-73.86480,40.87986,"b'\x01\x01\x00\x00\x00,e\x19\xe2XwR\xc0\x7f\x8...",360050380003,36005038000,160802,36,5,38000,3,1963,614.0,102,145,111,9,24,78,53.551564,162.224876,9.173185,1.074337,6.115456,0.743772,0.495848,0.743772,1.074337,6.115456,0.743772,0.247924,0.743772,0.000000,0.247924,0.0,32.274792,141.058785,0.00000,52.890433,105.780867,0.0,0.000000,233.35,14.500000,36,005,038000,3
60,114489,LORRIES DELI GROCERY CORP,365 E 193RD ST,BRONX,10458,0,DELI-GROCERY,-73.89090,40.86311,b'\x01\x01\x00\x00\x00i\x00o\x81\x04yR\xc0\xdf...,360050399013,36005039901,155928,36,5,39901,3,1155,403.0,104,164,108,24,47,37,62.315863,172.601491,16.139360,7.173049,0.149439,3.586524,1.494385,3.735963,7.173049,0.149439,3.586524,0.298877,3.735963,0.000000,1.195508,0.0,39.883580,95.640653,0.00000,0.000000,95.640653,0.0,0.000000,241.40,14.500000,36,005,039901,3
45,79663,NH DELI & GROCERY CORP,901 E 217TH ST,BRONX,10469,0,DELI-GROCERY,-73.85758,40.88025,b'\x01\x01\x00\x00\x00\tm9\x97\xe2vR\xc0\xe9&1...,360050388002,36005038800,151435,36,5,38800,2,1674,548.0,119,210,8,0,6,2,31.053853,89.013957,0.425395,0.106349,0.319046,0.000000,0.000000,0.000000,0.106349,0.319046,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,30.771010,90.762331,0.00000,34.031620,68.063240,0.0,0.000000,168.98,11.333333,36,005,038800,2


## Master Data Sample

In [10]:
master_file_path = Path(".").resolve().parent / "data/final/master_data.parquet"
master_df = pd.read_parquet(str(master_file_path))

master_df.sample(5)

,agtno,busnm,busaddr,buscity,buszip,chain,bustype,longitude,latitude,geometry,bg_geoid,tract_geoid,aadt_count,aadt_road_name,aadt_functional_class,aadt_lanes,dist_to_read_m,TotPop,HH,R_LowWageWk,R_MedWageWk,TotEmp,E_LowWageWk,E_MedWageWk,E_HiWageWk,D1A,D1B,D1C,D1C5_RET,D1C5_OFF,D1C5_IND,D1C5_SVC,D1C5_ENT,D1C8_RET,D1C8_OFF,D1C8_IND,D1C8_SVC,D1C8_ENT,D1C8_ED,D1C8_HLTH,D1C8_PUB,D3A,D3B,D3BAO,D3BMM3,D3BMM4,D3BPO3,D3BPO4,D4A,NatWalkInd,gas_station_within_0.5_mi,gas_station_within_1_mi,arts_and_entertainment_within_0.5_mi,arts_and_entertainment_within_1_mi,shopping_within_0.5_mi,shopping_within_1_mi,social_or_community_service_within_0.5_mi,social_or_community_service_within_1_mi,restaurant_within_0.5_mi,restaurant_within_1_mi,bar_within_0.5_mi,bar_within_1_mi,rac_c000,rac_ce01,rac_ce02,rac_ce03,wac_c000,wac_cns07,wac_cns18,wac_ce01,wac_ce02,wac_ce03
11263,106786,LEE STOP 1 MARKET CORP,55-20 103RD ST,CORONA,11368,0,DELI-GROCERY,-73.85657,40.74045,b'\x01\x01\x00\x00\x00\x06/\xfa\n\xd2vR\xc0_)\...,360810439002,36081043900,NaN,I495 SR WEST,19,3:0||||,397.670324,2186.0,633.0,165.0,319.0,434.0,60.0,120.0,254.0,30.402257,100.391743,19.931389,1.331821,0.137775,1.928844,14.971504,1.561445,1.331821,0.137775,1.928844,0.367399,1.561445,14.420406,0.183699,0.000000,38.244601,156.805841,0.000000,146.959551,58.783820,0.000000,0.000000,190.44,15.500000,4.0,7.0,0.0,1.0,0.0,8.0,6.0,22.0,3.0,22.0,1.0,10.0,606.0,117.0,220.0,269.0,430.0,20.0,37.0,31.0,102.0,297.0
7327,75502,MATEO COMMUNICATIONS,29 LAKE ST,MONROE,10950,0,MISCELLANEOUS,-74.18880,41.32854,b'\x01\x01\x00\x00\x00\x07_\x98L\x15\x8cR\xc0\...,360710141012,36071014101,6062.0,Lake St,19,1:1||||,2.876476,1795.0,502.0,102.0,159.0,1615.0,685.0,526.0,404.0,1.612025,5.764113,5.186096,1.300538,0.587650,0.330754,2.495106,0.472047,1.300538,0.305064,0.330754,0.703254,0.472047,0.128448,1.663404,0.282586,20.959426,114.626506,0.000000,20.551090,5.604843,95.282326,31.760775,278.95,17.666667,1.0,3.0,0.0,0.0,0.0,1.0,4.0,4.0,1.0,1.0,0.0,1.0,735.0,117.0,198.0,420.0,1694.0,380.0,110.0,449.0,559.0,686.0
14142,113443,J & H EXPRESS SWAN LAKE,1658 BRISCOE RD,SWAN LAKE,12783,0,CONV. STORE,-74.78291,41.75306,b'\x01\x01\x00\x00\x00\x87m\x8b2\x1b\xb2R\xc0\...,361059506003,36105950600,NaN,NaN,NaN,NaN,NaN,469.0,151.0,49.0,80.0,64.0,11.0,41.0,12.0,0.204454,0.157972,0.021557,0.000000,0.000000,0.002021,0.014147,0.005389,0.000000,0.000000,0.002021,0.012126,0.005389,0.000000,0.002021,0.000000,3.805829,7.386872,0.430107,2.795695,0.430107,7.311818,0.215053,-99999.00,5.833333,2.0,2.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,213.0,39.0,65.0,109.0,42.0,0.0,9.0,12.0,16.0,14.0
3140,77126,RITE AID #10839,1030 PINE AVE,NIAGARA FALLS,14301,296,DRUG STORE,-79.04966,43.09485,"b'\x01\x01\x00\x00\x00\xd7\xfa""\xa1-\xc3S\xc0G...",360630209002,36063020900,224.0,13TH ST,19,1:1||||,950.224226,562.0,505.0,94.0,128.0,290.0,149.0,111.0,30.0,8.032106,7.989458,4.122674,0.881399,0.199026,0.440700,2.217714,0.383835,0.881399,0.199026,0.440700,0.184810,0.383835,0.000000,2.032905,0.000000,28.936927,197.196871,0.000000,54.589888,9.098315,145.573034,54.589888,0.00,17.833333,2.0,5.0,0.0,2.0,0.0,5.0,6.0,18.0,2.0,15.0,0.0,6.0,281.0,70.0,87.0,124.0,359.0,51.0,16.0,137.0,159.0,63.0
4482,37691,EXIT 9 WINE & LIQUOR,9 HALFMOON CROSSING BLVD,CLIFTON PARK,12065,0,LIQUOR,-73.76905,42.85863,b'\x01\x01\x00\x00\x00H\xbf}\x1d8qR\xc0\xa7\xa...,360910624102,36091062410,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,3.0,7.0,0.0,0.0,2.0,3.0,0.0,3.0,1.0,7.0,0.0,2.0,889.0,140.0,202.0,547.0,1525.0,273.0,186.0,261.0,408.0,856.0
